In [1]:
import os
import json
import logging
import PyPDF2
import cohere
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
import gradio as gr

# Professional logging setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Libraries imported and logging configured.")

c:\Users\ASUS.SXT412\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



✅ Libraries imported and logging configured.


In [ ]:
COHERE_API_KEY = "" # ⚠️ Replace with your actual key
PDF_FOLDER = "data"
LOG_FILE = "chat_history_logs.jsonl"

try:
    cohere_client = cohere.Client(COHERE_API_KEY)
    logger.info("Cohere client initialized.")
except Exception as e:
    logger.error(f"Failed to initialize Cohere: {e}")

2026-03-13 12:22:45,710 - INFO - Cohere client initialized.


In [3]:
# ===============================
# 3) LOAD PDFs AND CREATE DATABASE
# ===============================
def setup_knowledge_base(folder):
    if not os.path.exists(folder):
        os.makedirs(folder)
        logger.warning(f"Created '{folder}' directory. Please add PDFs.")
        return [], None, None

    raw_texts = []
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            try:
                with open(os.path.join(folder, file), "rb") as f:
                    reader = PyPDF2.PdfReader(f)
                    for page in reader.pages:
                        content = page.extract_text()
                        if content: 
                            raw_texts.append(content)
            except Exception as e:
                logger.error(f"Error reading {file}: {e}")
                
    if not raw_texts:
        logger.warning("No text extracted from PDFs.")
        return [], None, None

    # Chunking
    chunks = []
    for text in raw_texts:
        words = text.split()
        for i in range(0, len(words), 300):
            chunks.append(" ".join(words[i:i+300]))

    # Embedding
    logger.info("Loading embedding model...")
    embed_model = SentenceTransformer("paraphrase-MiniLM-L3-v2")
    embeddings = embed_model.encode(chunks, normalize_embeddings=True).astype("float32")
    
    # Vector DB
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    logger.info(f"Vector database created with {len(chunks)} chunks.")
    
    return chunks, embed_model, index

chunks, embed_model, index = setup_knowledge_base(PDF_FOLDER)

2026-03-13 12:22:54,902 - INFO - Loading embedding model...
2026-03-13 12:22:54,904 - INFO - Use pytorch device_name: cpu
2026-03-13 12:22:54,904 - INFO - Load pretrained SentenceTransformer: paraphrase-MiniLM-L3-v2
Batches: 100%|██████████| 34/34 [00:06<00:00,  4.86it/s]
2026-03-13 12:23:04,215 - INFO - Vector database created with 1059 chunks.


In [4]:
# ===============================
# 4) CONVERSATIONAL CHAT ENGINE
# ===============================
def save_interaction(user_input, bot_output):
    """Saves conversation data in a professional JSON Lines format."""
    entry = {"user": user_input, "bot": bot_output}
    try:
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry) + "\n")
    except Exception as e:
        logger.error(f"Failed to save log: {e}")

def chat_engine(message, history):
    if index is None or embed_model is None:
        return "I don't have any documents loaded right now. Please add some PDFs to the 'my_documents' folder and restart me!"

    # 1. Retrieve Context
    q_vec = embed_model.encode([message], normalize_embeddings=True).astype("float32")
    distances, ids = index.search(q_vec, k=4)
    context = "\n---\n".join([chunks[i] for i in ids[0]])

    # 2. Format History for Cohere (Safe format for all Gradio versions)
    cohere_history = []
    for chat_pair in history:
        if isinstance(chat_pair, (list, tuple)) and len(chat_pair) == 2:
            cohere_history.append({"role": "USER", "message": chat_pair[0]})
            cohere_history.append({"role": "CHATBOT", "message": chat_pair[1]})

    # 3. Professional Conversational Preamble (Brain & Safety)
    system_prompt = """You are a highly professional, intelligent, and conversational AI assistant. 
    Your primary goal is to help the user by answering questions based on the provided 'Context' documents.
    
    Guidelines:
    1. Be Conversational: If the user greets you or makes small talk, respond naturally and politely before addressing their core question.
    2. Document Integration: Seamlessly weave facts from the context into your conversational replies. Don't just spit out raw data.
    3. Honesty: If the answer is not in the context, politely inform the user that the provided documents don't contain that specific information, but offer to help with anything else.
    4. Safety Guardrails: You must strictly refuse any requests related to hacking, cheating, bypassing security, or illegal activities. Politely state that you cannot assist with such topics.
    """

    prompt = f"Context from documents:\n{context}\n\nUser's message:\n{message}"
    
    # 4. Generate Response
    try:
        response = cohere_client.chat(
            message=prompt,
            chat_history=cohere_history,
            preamble=system_prompt,
            model="command-r-08-2024",
            max_tokens=400,
            temperature=0.4 # Higher temperature makes it sound more human
        )
        bot_reply = response.text
    except Exception as e:
        logger.error(f"Cohere API Error: {e}")
        return "I'm sorry, I encountered an error while connecting to my AI core. Please check your API key."
    
    # 5. Log the interaction
    save_interaction(message, bot_reply)
    
    return bot_reply

logger.info("Chat engine ready.")

2026-03-13 12:23:04,230 - INFO - Chat engine ready.


In [ ]:
# ===============================
# 5) PROFESSIONAL GRADIO UI
# ===============================
logger.info("Launching Chat Interface...")

chatbot_ui = gr.Chatbot(
    avatar_images=("👤", "🤖"),
    height=500
)

with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("# Corporate Document Assistant")
    gr.Markdown("Ask me anything about your uploaded documents.")

    gr.ChatInterface(
        fn=chat_engine,
        chatbot=chatbot_ui,
        examples=[
            "Hello! How are you?",
            "Can you summarize the main points?",
            "What data is missing from these files?"
        ]
    )

app.launch()

2026-03-13 12:23:04,241 - INFO - Launching Chat Interface...
C:\Users\ASUS.SXT412\AppData\Local\Temp\ipykernel_32044\1791366164.py:11: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


* Running on local URL:  http://127.0.0.1:7860


2026-03-13 12:23:05,156 - INFO - HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-13 12:23:05,436 - INFO - HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


2026-03-13 12:23:06,105 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-13 12:23:08,859 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
